# NBA Playoff Predictor — Kaggle (P100 GPU)

Clones https://github.com/ferhat00/nba-playoffs (main), applies runtime CUDA monkey-patches, and runs the full pipeline.

**Before running:** in Kaggle notebook settings set **Accelerator = GPU P100** and **Internet = On** (`nba_api` needs network; fallback data exists but real data is better).

### Why P100 (not T4x2, not TPU)
The critical path is PyMC MCMC on the Bayesian player evaluator; switching to `nuts_sampler="numpyro"` gives us JAX/CUDA. P100's HBM2 bandwidth and FP64 throughput matter for NUTS leapfrog steps; T4's FP16 strengths don't apply. numpyro doesn't auto-shard 2 chains across 2 T4s, so the second T4 would sit idle. TPU would require a JAX rewrite for the PyTorch LSTM and XGBoost — effort far exceeds the gain for a ~2–5 min pipeline.

### What we accelerate
- **PyMC MCMC** → `nuts_sampler="numpyro"` (biggest win)
- **PyTorch LSTM** → `.to("cuda")` (small model; modest gain)
- **XGBoost** → **left on CPU** (only ~2500 rows; CUDA is slower here)
- **Monte Carlo brackets** → already parallelized via multiprocessing

## Cell 1 — Environment check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import sys, platform
print(sys.version)
print(platform.platform())

## Cell 2 — Install JAX + numpyro for GPU

Kaggle's GPU image ships CUDA 12.x, PyTorch, and XGBoost preinstalled. We only add JAX+numpyro (for PyMC's GPU sampler) and pin PyMC/ArviZ to the versions in `requirements.txt`.

In [ ]:
!pip install -q --upgrade "jax[cuda12]" numpyro
!pip install -q "pymc>=5.10,<6.0" "arviz>=0.17,<1.0" "nba_api>=1.4,<2.0" "plotly>=5.18,<6.0"

# P100 (sm_60) support was dropped in PyTorch 2.5.
# If the pre-installed version is >=2.5, downgrade to the last compatible build.
import torch
_major, _minor = (int(x) for x in torch.__version__.split(".")[:2])
if (_major, _minor) >= (2, 5):
    print(f"PyTorch {torch.__version__} does not support P100 (sm_60) — downgrading to 2.4.1")
    !pip install -q torch==2.4.1 --index-url https://download.pytorch.org/whl/cu121
else:
    print(f"PyTorch {torch.__version__} supports P100 — no downgrade needed")

## Cell 3 — Clone repo (main branch)

In [ ]:
import os, sys, shutil
REPO_DIR = "/kaggle/working/nba-playoffs"
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone --depth 1 --branch main https://github.com/ferhat00/nba-playoffs.git {REPO_DIR}
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

## Cell 4 — Verify CUDA is visible to PyTorch and JAX

In [ ]:
import torch, jax
print("torch.cuda.is_available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("  device:", torch.cuda.get_device_name(0))
print("jax.devices:", jax.devices())

## Cell 5 — Runtime monkey-patches

All modifications are in-process (no file edits in the cloned repo). This keeps the notebook diffable and survives future upstream changes.

- **(a)** Wrap `pm.sample` in `player_evaluator` so the existing call picks up `nuts_sampler="numpyro"`.
- **(b)** In `TeamStateAggregator.build`, shim `torch.from_numpy` and the `_LSTMMomentum` class so tensors and model land on CUDA for the training loop.
- **(c)** Force multiprocessing `spawn` start method to avoid fork-after-CUDA-init corruption when `bracket_simulator` forks workers.
- **(d)** XGBoost is intentionally left on CPU (`device="cuda"` is slower on 2500 rows).

In [ ]:
# (a) PyMC -> numpyro
import sys, os
REPO_DIR = "/kaggle/working/nba-playoffs"
if REPO_DIR not in sys.path:
    if not os.path.isdir(REPO_DIR):
        raise RuntimeError(f"{REPO_DIR} not found — run Cell 3 (clone repo) first")
    sys.path.insert(0, REPO_DIR)

from nba_playoff_predictor.models import player_evaluator as pe
import pymc as pm

_orig_sample = pm.sample
def _gpu_sample(*args, **kwargs):
    kwargs.setdefault("nuts_sampler", "numpyro")
    kwargs.setdefault("progressbar", False)
    return _orig_sample(*args, **kwargs)
pe.pm.sample = _gpu_sample
print("patched: player_evaluator.pm.sample -> numpyro")

In [ ]:
# (b) PyTorch LSTM -> CUDA (with compatibility probe)
from nba_playoff_predictor.models import team_aggregator as ta
import torch

_DEVICE = torch.device("cpu")
if torch.cuda.is_available():
    try:
        # Probe: launch a trivial kernel to verify sm_XX compatibility
        torch.tensor([1.0], device="cuda").sum()
        _DEVICE = torch.device("cuda")
    except RuntimeError as e:
        print(f"⚠ PyTorch CUDA kernels incompatible with this GPU — LSTM stays on CPU")
        print(f"  ({e})")

_orig_build = ta.TeamStateAggregator.build

def _gpu_build(self, *a, **kw):
    real_from_numpy = ta.torch.from_numpy
    real_lstm_cls = ta._LSTMMomentum

    def _from_numpy_cuda(arr):
        return real_from_numpy(arr).to(_DEVICE)

    class _LSTMCuda(real_lstm_cls):
        def __init__(self, *aa, **kk):
            super().__init__(*aa, **kk)
            self.to(_DEVICE)

    ta.torch.from_numpy = _from_numpy_cuda
    ta._LSTMMomentum = _LSTMCuda
    try:
        return _orig_build(self, *a, **kw)
    finally:
        ta.torch.from_numpy = real_from_numpy
        ta._LSTMMomentum = real_lstm_cls

ta.TeamStateAggregator.build = _gpu_build
print("patched: TeamStateAggregator.build -> device =", _DEVICE)

In [ ]:
# (c) Multiprocessing safety: avoid fork-after-CUDA-init
import multiprocessing as mp
try:
    mp.set_start_method("spawn", force=True)
    print("mp start method ->", mp.get_start_method())
except RuntimeError as e:
    print("mp start method already set:", mp.get_start_method(), "|", e)

## Cell 6 — Run the pipeline (in-process)

We import and call `run()` directly instead of `!python -m nba_playoff_predictor.main`. A subprocess would not inherit the monkey-patches above.

Run once with `QUICK = True` first (<60s smoke test), then flip to `QUICK = False` for the full 100k-sim run.

In [ ]:
from nba_playoff_predictor.main import run, _configure_logging, _set_seeds

_configure_logging(verbose=True)
_set_seeds(42)

QUICK = True  # set False for the full 100k-sim / 500-MCMC-draw run

if QUICK:
    run(n_simulations=1_000, mcmc_samples=100)
else:
    run(n_simulations=100_000, mcmc_samples=500)

## Cell 7 — Display outputs

In [ ]:
from pathlib import Path
from IPython.display import Image, HTML, display

out_dir = Path(REPO_DIR) / "output"
if not out_dir.exists():
    print("No output/ directory yet — did the run complete?")
else:
    for p in sorted(out_dir.glob("*")):
        print(p)
        if p.suffix.lower() in (".png", ".jpg", ".jpeg"):
            display(Image(str(p)))
        elif p.suffix.lower() == ".html":
            display(HTML(p.read_text()))

## Cell 8 (optional) — Quick wall-clock benchmark

Sanity check that the GPU path is actually helping. For the small `--quick` config, JAX compile warmup can erase the MCMC speedup — expect GPU to pay off at the full 500-draw config.

In [ ]:
import time
t = time.perf_counter()
run(n_simulations=1_000, mcmc_samples=100)
print(f"quick run wall-clock: {time.perf_counter() - t:.1f}s")